In [ ]:
# ── Imports ───────────────────────────────────────────────────
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from scipy.stats import t as student_t, shapiro

from src.preprocessing_btc import load_data

# ── Load data ─────────────────────────────────────────────────
params, series = load_data()

print(params)
print(f'Data period: {series.index[0]} to {series.index[-1]}')
print(f'Total observations: {len(series)}')
print(series.head())

In [ ]:
# ── Plot 1: Raw Bitcoin prices ────────────────────────────────
import os
os.makedirs('../../outputs/figures/btc', exist_ok=True)

raw = pd.read_csv('../../data/raw/btc_prices_monthly_2014m09_2026m05.csv')
raw['Date'] = pd.to_datetime(raw['Date'].str[:7], format='%Y-%m')
raw = raw.set_index('Date')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(raw.index, raw['BTC_Price_USD'], color='steelblue', linewidth=0.8)
ax.set_title('Monthly Bitcoin Prices (2014-2026)', fontsize=13)
ax.set_ylabel('USD')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../../outputs/figures/btc/01_raw_btc_prices.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: HP-filtered cycle ─────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.index, series['cycle'], color='steelblue', linewidth=1)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(pd.Timestamp('2021-11-01'), color='red', linewidth=1,
           linestyle='--', label='Bubble peak (Nov 2021)')
ax.axvline(pd.Timestamp('2025-10-01'), color='orange', linewidth=1,
           linestyle='--', label='Bubble peak (Oct 2025)')
ax.set_ylabel('Deviation from trend (USD)')
ax.set_title('HP-Filtered Bitcoin Prices - Cyclical Component (2014-2024)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig('../../outputs/figures/btc/02_hp_filtered_cycle.png', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Noncausal component u_t ──────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(series.index, series['u_t'], color='steelblue', linewidth=1)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(pd.Timestamp('2021-11-01'), color='red', linewidth=1,
           linestyle='--', label='Bubble peak (Nov 2021)')
ax.axvline(pd.Timestamp('2025-10-01'), color='orange', linewidth=1,
           linestyle='--', label='Bubble peak (Oct 2025)')
ax.set_ylabel('$u_t$')
ax.set_title('Noncausal Component $u_t$ - Bitcoin Prices (2014-2024)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.legend()
plt.tight_layout()
plt.savefig('../../outputs/figures/btc/03_noncausal_component.png', dpi=150)
plt.show()

## Parameter Estimates

MAR(1,1) estimated via approximate MLE with Student-t errors using the MARX package.

AIC selects p=2. Among all MAR(r,s) with r+s=2, AIC selects MAR(0,2) but psi1=1.123
exceeds unity, making the bubble dynamics economically uninterpretable within the MAR
framework. MAR(1,1) is selected as the best economically interpretable specification at p=2.

In [ ]:
# ── AIC model comparison table ────────────────────────────────
model_comparison = pd.DataFrame({
    'Model':  ['MAR(2,0)', 'MAR(1,1)', 'MAR(0,2)'],
    'LL':     [-1188.71,   -1176.44,   -1164.12],
    'AIC':    [2385.41,    2360.87,    2336.24],
    'phi':    ['0.9677',   '0.5129',   '0'],
    'psi':    ['0',        '0.6888',   '1.1230, -0.2027'],
    'Note':   ['',         'Selected', 'psi1 > 1']
})

print('AIC Model Comparison (p=2):')
print(model_comparison.to_string(index=False))

In [ ]:
# ── Parameter estimates table ─────────────────────────────────
estimates = pd.DataFrame({
    'Parameter':    ['phi (causal)', 'psi (noncausal)', 'df', 'scale'],
    'Bitcoin':      [
        f'{params.phi:.3f} ({params.phi_se:.3f})',
        f'{params.psi:.3f} ({params.psi_se:.3f})',
        f'{params.df:.3f}',
        f'{params.scale:.1f}'
    ],
    'Nickel (Hecq & Voisin 2021)': ['0.660 (0.017)', '0.730 (0.013)', '1.450', '390.0']
})

print('MAR(1,1) Parameter Estimates (standard errors in parentheses):')
print(estimates.to_string(index=False))

In [ ]:
# ── Residual time series ──────────────────────────────────────
residuals = series['residuals'].dropna()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(residuals.index, residuals.values, color='steelblue', linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('MAR(1,1) Residuals - Bitcoin', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Residual')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('../../outputs/figures/btc/04_residuals.png', dpi=150)
plt.show()

In [ ]:
# ── Residual distribution vs Student-t fit ────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(residuals.values, bins=50, density=True, color='steelblue',
        alpha=0.6, label='Residuals')

x = np.linspace(residuals.min(), residuals.max(), 500)
pdf = student_t.pdf(x, df=params.df, loc=0, scale=params.scale)
ax.plot(x, pdf, color='crimson', linewidth=2,
        label=f'Student-t (df={params.df:.2f}, scale={params.scale:.1f})')

ax.set_title('Residual Distribution vs Student-t Fit - Bitcoin')
ax.set_xlabel('Residual')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('../../outputs/figures/btc/05_residual_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Non-normality tests ───────────────────────────────────────
stat, p_value = shapiro(residuals.values[:500])
print('=== Shapiro-Wilk Test for Normality ===')
print(f'Statistic: {stat:.4f}')
print(f'P-value:   {p_value:.6f}')
print(f"Conclusion: {'Reject normality (p < 0.05)' if p_value < 0.05 else 'Cannot reject normality'}")
print()

kurt = stats.kurtosis(residuals.values)
skew = stats.skew(residuals.values)
print(f'Excess kurtosis: {kurt:.4f}')
print(f'Skewness:        {skew:.4f}')